# Step 4：二次隐含波动率与 Half Skew

本 Notebook 使用 Black–76 从期权收盘价反解IV，仅使用OTM样本，按“交易日 × 到期日”拟合

\[
\sigma(k)=a+bk+\tfrac12ck^2,\qquad k=\ln(K/F).
\]

给定外置参数 `HALF_SKEW_H=h`：

\[
CallSkew=\sigma(h)-\sigma(0),\quad
PutSkew=\sigma(-h)-\sigma(0),\quad
WingCurvature=\frac{CallSkew+PutSkew}{2}.
\]

04不再保留SVI、导数Skew或25Delta Skew接口。

## 参数与依赖

In [ ]:
_module_parameter_names = [
    'VOL_MODEL', 'HALF_SKEW_H', 'MIN_OTM_OPTIONS_PER_EXPIRY', 'MIN_OPTION_PRICE',
    'MIN_IV', 'MAX_IV', 'IV_SOLVER_LOWER_BOUND',
    'IV_SOLVER_TOLERANCE', 'IV_SOLVER_MAX_ITERATIONS', 'MODEL_SKEW_TAG',
    'VOLATILITY_MODEL_OUTPUT_PATH', 'IV_CURVE_FIGURE_PATH', 'SAVE_CSV',
    'SAVE_FIGURE', 'FIGURE_FORMAT',
]
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')

In [ ]:
import math
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    warnings.warn('matplotlib未安装，将跳过IV曲线图片', RuntimeWarning)

_required_config = {
    'VOL_MODEL', 'HALF_SKEW_H', 'MIN_OTM_OPTIONS_PER_EXPIRY', 'MIN_OPTION_PRICE',
    'MIN_IV', 'MAX_IV', 'IV_SOLVER_LOWER_BOUND',
    'IV_SOLVER_TOLERANCE', 'IV_SOLVER_MAX_ITERATIONS', 'MODEL_SKEW_TAG',
    'VOLATILITY_MODEL_OUTPUT_PATH', 'IV_CURVE_FIGURE_PATH', 'SAVE_CSV',
    'SAVE_FIGURE', 'FIGURE_FORMAT',
}
_required_functions = {'black76_price', 'is_otm_option', '_normalize_option_type'}
_ipython = get_ipython() if 'get_ipython' in globals() else None
if not _required_config.issubset(globals()):
    if _ipython is None: raise RuntimeError('请先运行00_config.ipynb')
    _ipython.run_line_magic('run', './00_config.ipynb')
if str(VOL_MODEL).upper() != 'QUADRATIC':
    raise ValueError("04固定要求VOL_MODEL='QUADRATIC'")
if not _required_functions.issubset(globals()):
    if _ipython is None: raise RuntimeError('请先运行02_basic_functions.ipynb')
    _ipython.run_line_magic('run', './02_basic_functions.ipynb')
if 'option_forward_panel' not in globals():
    if _ipython is None: raise RuntimeError('请先运行03_repo_forward.ipynb')
    _ipython.run_line_magic('run', './03_repo_forward.ipynb')

## Black–76 IV反解与二次拟合

In [ ]:
def black76_implied_volatility(
    price: float, F: float, K: float, tau: float, r: float, option_type: str,
    lower_vol: float = 1e-6, upper_vol: float = 5.0,
    tolerance: float = 1e-8, max_iterations: int = 200,
) -> float:
    values = [price, F, K, tau, r, lower_vol, upper_vol]
    if not all(np.isfinite(values)) or min(price, F, K, tau) <= 0:
        raise ValueError('INVALID_INPUT')
    option_type = _normalize_option_type(option_type)
    discount = math.exp(-r * tau)
    lower_price = discount * max(F-K, 0.0) if option_type == 'CALL' else discount * max(K-F, 0.0)
    upper_price = discount * (F if option_type == 'CALL' else K)
    price_tolerance = max(tolerance, 1e-12 * max(1.0, upper_price))
    if price <= lower_price + price_tolerance: raise ValueError('AT_OR_BELOW_INTRINSIC')
    if price >= upper_price - price_tolerance: raise ValueError('AT_OR_ABOVE_UPPER_BOUND')
    low, high = lower_vol, upper_vol
    if not (black76_price(F,K,low,tau,r,option_type) <= price <= black76_price(F,K,high,tau,r,option_type)):
        raise ValueError('OUTSIDE_IV_SEARCH_RANGE')
    for _ in range(max_iterations):
        mid = 0.5 * (low + high)
        model_price = black76_price(F,K,mid,tau,r,option_type)
        if abs(model_price-price) <= tolerance: return float(mid)
        if model_price < price: low = mid
        else: high = mid
    return float(0.5*(low+high))


def calculate_option_implied_vols(panel, lower_vol, upper_vol, tolerance, max_iterations,
                                  min_option_price=None, min_iv=None, max_iv=None):
    rows = []
    for row in panel.itertuples(index=False):
        result = row._asdict()
        result.update(LOG_MONEYNESS=np.nan, IS_OTM=False, IV=np.nan, IV_STATUS='INVALID_INPUT')
        try:
            if min_option_price is not None and row.PRICE < min_option_price: raise ValueError('LOW_PRICE')
            k = math.log(row.STRIKE/row.FORWARD)
            iv = black76_implied_volatility(row.PRICE,row.FORWARD,row.STRIKE,row.TAU,
                                            row.RISK_FREE_RATE,row.TYPE,lower_vol,upper_vol,
                                            tolerance,max_iterations)
            if min_iv is not None and iv < min_iv: raise ValueError('IV_BELOW_MIN')
            if max_iv is not None and iv > max_iv: raise ValueError('IV_ABOVE_MAX')
            result.update(LOG_MONEYNESS=float(k), IS_OTM=bool(is_otm_option(row.FORWARD,row.STRIKE,row.TYPE)),
                          IV=float(iv), IV_STATUS='OK')
        except (TypeError, ValueError, OverflowError) as exc:
            result['IV_STATUS'] = str(exc)
        rows.append(result)
    return pd.DataFrame(rows).sort_values(['TRADE_DT','EXPIRY','STRIKE','TYPE']).reset_index(drop=True)


def fit_quadratic_model(k_values, iv_values):
    k, iv = np.asarray(k_values,float), np.asarray(iv_values,float)
    if len(k) < 3: raise ValueError('二次模型至少需要3个样本')
    design = np.column_stack([np.ones_like(k), k, 0.5*k*k])
    params, _, rank, _ = np.linalg.lstsq(design, iv, rcond=None)
    fitted, status = design@params, 'QUADRATIC'
    if rank < 3 or np.any(~np.isfinite(fitted)) or np.any(fitted <= 0):
        linear = np.column_stack([np.ones_like(k),k])
        lp, _, _, _ = np.linalg.lstsq(linear,iv,rcond=None)
        params, status = np.array([lp[0],lp[1],0.0]), 'FALLBACK_LINEAR'
        fitted = design@params
    if np.any(~np.isfinite(fitted)) or np.any(fitted <= 0):
        params, status = np.array([float(np.mean(iv)),0.0,0.0]), 'FALLBACK_CONSTANT'
        fitted = design@params
    return {'a':float(params[0]),'b':float(params[1]),'c':float(params[2]),
            'RMSE':float(np.sqrt(np.mean((fitted-iv)**2))),'FIT_STATUS':status}


def fit_daily_volatility_models(iv_panel, min_samples):
    valid = iv_panel.loc[iv_panel.IV_STATUS.eq('OK') & iv_panel.IS_OTM].copy()
    rows = []
    for (trade_date,expiry), group in valid.groupby(['TRADE_DT','EXPIRY'],sort=True):
        first = group.iloc[0]
        base = {'TRADE_DT':trade_date,'EXPIRY':expiry,'EXPIRY_CODE':first.EXPIRY_CODE,
                'TAU':float(first.TAU),'FORWARD':float(first.FORWARD),
                'RISK_FREE_RATE':float(first.RISK_FREE_RATE),'MODEL':'QUADRATIC','N_SAMPLES':len(group)}
        if len(group) < min_samples:
            rows.append({**base,'FIT_STATUS':'INSUFFICIENT_SAMPLES','RMSE':np.nan})
            continue
        try: rows.append({**base,**fit_quadratic_model(group.LOG_MONEYNESS,group.IV)})
        except Exception as exc: rows.append({**base,'FIT_STATUS':f'FAILED: {exc}','RMSE':np.nan})
    return pd.DataFrame(rows).sort_values(['TRADE_DT','EXPIRY']).reset_index(drop=True)


def evaluate_volatility_model(model_row, k_values):
    k = np.asarray(k_values,float)
    values = float(model_row['a']) + float(model_row['b'])*k + 0.5*float(model_row['c'])*k*k
    return np.where(np.isfinite(values) & (values > 0), values, np.nan)

## Half Skew锚点与指标

In [ ]:
def build_half_skew_metrics(model_parameters, h):
    if h <= 0: raise ValueError('HALF_SKEW_H必须为正数')
    rows, landmarks = [], []
    successful = model_parameters.loc[~model_parameters.FIT_STATUS.astype(str).str.startswith(('FAILED','INSUFFICIENT'))]
    for _, p in successful.iterrows():
        atm = float(evaluate_volatility_model(p,0.0))
        call_iv = float(evaluate_volatility_model(p,h))
        put_iv = float(evaluate_volatility_model(p,-h))
        call_skew, put_skew = call_iv-atm, put_iv-atm
        wing_curvature = 0.5*(call_skew+put_skew)
        common = {'TRADE_DT':p.TRADE_DT,'EXPIRY':p.EXPIRY,'EXPIRY_CODE':p.EXPIRY_CODE,
                  'MODEL':'QUADRATIC','HALF_SKEW_H':h}
        rows.append({**common,'ATM_VOL':atm,'CALL_IV':call_iv,'PUT_IV':put_iv,
                     'CALL_SKEW':call_skew,'PUT_SKEW':put_skew,
                     'WING_CURVATURE':wing_curvature})
        for label,k,iv in [('PUT_ANCHOR',-h,put_iv),('ATM',0.0,atm),('CALL_ANCHOR',h,call_iv)]:
            landmarks.append({**common,'TARGET':label,'LOG_MONEYNESS':k,
                              'STRIKE':float(p.FORWARD)*math.exp(k),'IV':iv,'STATUS':'OK'})
    metrics = pd.DataFrame(rows).sort_values(['TRADE_DT','EXPIRY']).reset_index(drop=True)
    anchors = pd.DataFrame(landmarks).sort_values(['TRADE_DT','EXPIRY','LOG_MONEYNESS']).reset_index(drop=True)
    return metrics, anchors


def build_skew_panel(half_skew_metrics):
    # 兼容下游读取习惯；不再提供含义模糊的单列SKEW。
    columns = ['TRADE_DT','EXPIRY','EXPIRY_CODE','MODEL','HALF_SKEW_H','ATM_VOL',
               'CALL_SKEW','PUT_SKEW','WING_CURVATURE']
    return half_skew_metrics[columns].copy()

## 每日多期限IV曲线

In [ ]:
def plot_iv_curves(option_iv_panel, model_parameters, half_skew_metrics, figure_path,
                   h, figure_format='png'):
    if plt is None: return []
    figure_path = Path(figure_path); figure_path.mkdir(parents=True,exist_ok=True)
    valid = option_iv_panel.loc[option_iv_panel.IV_STATUS.eq('OK') & option_iv_panel.IS_OTM]
    successful = model_parameters.loc[~model_parameters.FIT_STATUS.astype(str).str.startswith(('FAILED','INSUFFICIENT'))]
    saved = []
    for trade_date, daily in successful.groupby('TRADE_DT',sort=True):
        daily = daily.sort_values('EXPIRY').reset_index(drop=True)
        ncols, nrows = 2, int(math.ceil(len(daily)/2))
        fig, axes = plt.subplots(nrows,ncols,figsize=(14,4.7*nrows+1),squeeze=False)
        fig.suptitle(f'MO OTM IV Smile - {trade_date:%Y%m%d} | Half Skew h={h:.3f}',
                     fontsize=15,fontweight='bold',x=.06,ha='left')
        for i,(_,p) in enumerate(daily.iterrows()):
            ax = axes.flat[i]
            group = valid.loc[valid.TRADE_DT.eq(trade_date)&valid.EXPIRY.eq(p.EXPIRY)]
            metric = half_skew_metrics.loc[half_skew_metrics.TRADE_DT.eq(trade_date)&half_skew_metrics.EXPIRY.eq(p.EXPIRY)].iloc[0]
            kmin = min(float(group.LOG_MONEYNESS.min()),-h)-.025
            kmax = max(float(group.LOG_MONEYNESS.max()), h)+.025
            kgrid = np.linspace(kmin,kmax,400)
            strike_grid = float(p.FORWARD)*np.exp(kgrid)
            for cp,color in [('PUT','#ef5350'),('CALL','#3f7de8')]:
                sample=group.loc[group.TYPE.eq(cp)]
                ax.scatter(sample.STRIKE,sample.IV,s=20,color=color,alpha=.95,label=cp)
            ax.plot(strike_grid,evaluate_volatility_model(p,kgrid),color='#172033',lw=1.8,label='Quadratic fit')
            for label,k,color in [('Put anchor',-h,'#d1495b'),('ATM',0.0,'#55789e'),('Call anchor',h,'#2b6cb0')]:
                strike=float(p.FORWARD)*math.exp(k)
                ax.axvline(strike,color=color,ls='--',lw=1.15,alpha=.9)
                ax.text(strike,.985,f'{label}\nk={k:+.3f}',transform=ax.get_xaxis_transform(),ha='center',va='top',fontsize=7)
            metric_text=(f"Call skew={metric.CALL_SKEW:.2%} | Put skew={metric.PUT_SKEW:.2%} | "
                         f"Wing curvature={metric.WING_CURVATURE:.2%}")
            ax.set_title(f"MO{p.EXPIRY_CODE} exp {p.EXPIRY:%Y-%m-%d} | a={p.a:.2%} b={p.b:.4f} c={p.c:.4f}\n"
                         f"{metric_text}\nsigma(k)=a+b*k+0.5*c*k^2 | N={int(p.N_SAMPLES)} | RMSE={p.RMSE:.2%}",
                         loc='left',fontsize=8.5,pad=10)
            ax.set_xlabel('Strike K'); ax.set_ylabel('IV')
            ax.yaxis.set_major_formatter(lambda value,pos:f'{value:.1%}')
            ax.grid(color='#d9e0e8',lw=.7,alpha=.8); ax.tick_params(labelsize=7)
        for i in range(len(daily),nrows*ncols): axes.flat[i].axis('off')
        fig.tight_layout(rect=[0,0,1,.97],h_pad=2.5,w_pad=1.8)
        output=figure_path/f'{trade_date:%Y%m%d}_all_expiries.{figure_format}'
        fig.savefig(output,dpi=160,bbox_inches='tight',facecolor='white'); plt.close(fig); saved.append(output)
    print(f'IV curve figures saved: {len(saved)}')
    return saved

## 质量检查、保存与统一入口

In [ ]:
def run_volatility_model_quality_checks(iv_panel, parameters, metrics, anchors, min_samples):
    success = ~parameters.FIT_STATUS.astype(str).str.startswith(('FAILED','INSUFFICIENT'))
    checks = [
        ('存在有效IV',iv_panel.IV_STATUS.eq('OK').any()),
        ('存在OTM拟合样本',(iv_panel.IV_STATUS.eq('OK')&iv_panel.IS_OTM).any()),
        ('存在成功二次模型',success.any()),
        ('成功模型样本充分',parameters.loc[success,'N_SAMPLES'].ge(min_samples).all()),
        ('成功模型RMSE有限',np.isfinite(parameters.loc[success,'RMSE']).all()),
        ('Half-Skew指标完整',metrics[['ATM_VOL','CALL_SKEW','PUT_SKEW','WING_CURVATURE']].notna().all().all()),
        ('每个截面有三个锚点',anchors.groupby(['TRADE_DT','EXPIRY']).size().eq(3).all()),
        ('Wing Curvature恒等式',np.allclose(metrics.WING_CURVATURE,.5*(metrics.CALL_SKEW+metrics.PUT_SKEW))),
    ]
    table=pd.DataFrame(checks,columns=['CHECK','PASSED'])
    if not table.PASSED.all(): warnings.warn(f'质量检查失败: {table.loc[~table.PASSED,"CHECK"].tolist()}')
    display(table); return table


def save_volatility_model_results(iv_panel,parameters,anchors,metrics,skew_panel,output_path):
    output_path=Path(output_path); output_path.mkdir(parents=True,exist_ok=True)
    frames={
        'option_iv_panel':iv_panel,
        'volatility_model_parameters':parameters,
        'half_skew_landmarks':anchors,
        'half_skew_metrics':metrics,
        'skew_panel':skew_panel,
    }
    files={name:output_path/f'{name}.csv' for name in frames}
    for name,frame in frames.items(): frame.to_csv(files[name],index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
    for name,path in files.items(): print(f'{name}: {path}')
    return files


def build_volatility_model_data(verbose=True):
    iv_panel=calculate_option_implied_vols(option_forward_panel,IV_SOLVER_LOWER_BOUND,MAX_IV,
                                           IV_SOLVER_TOLERANCE,IV_SOLVER_MAX_ITERATIONS,
                                           MIN_OPTION_PRICE,MIN_IV,MAX_IV)
    parameters=fit_daily_volatility_models(iv_panel,MIN_OTM_OPTIONS_PER_EXPIRY)
    metrics,anchors=build_half_skew_metrics(parameters,float(HALF_SKEW_H))
    skew=build_skew_panel(metrics)
    if verbose:
        print(iv_panel.IV_STATUS.value_counts().head(10)); display(parameters.head())
        display(metrics.head()); display(anchors.head(9))
    return iv_panel,parameters,anchors,metrics,skew

## 执行与结果

In [ ]:
option_iv_panel, volatility_model_parameters, half_skew_landmarks, half_skew_metrics, skew_panel = build_volatility_model_data(True)
volatility_model_quality_checks = run_volatility_model_quality_checks(
    option_iv_panel,volatility_model_parameters,half_skew_metrics,half_skew_landmarks,
    MIN_OTM_OPTIONS_PER_EXPIRY,
)
iv_curve_files = plot_iv_curves(
    option_iv_panel,volatility_model_parameters,half_skew_metrics,
    IV_CURVE_FIGURE_PATH,float(HALF_SKEW_H),FIGURE_FORMAT,
) if SAVE_FIGURE else []
volatility_model_files = save_volatility_model_results(
    option_iv_panel,volatility_model_parameters,half_skew_landmarks,
    half_skew_metrics,skew_panel,VOLATILITY_MODEL_OUTPUT_PATH,
) if SAVE_CSV else {}